In [2]:
import pyspark
from pyspark.sql import SparkSession

In [4]:
spark = SparkSession.builder.config('spark.driver.host','localhost').appName('06').getOrCreate()

In [5]:
bnb = spark.read.csv('C:\\datas\\3rd_week_data\\air_bnb.csv',header=True)

In [6]:
bnb.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: string (nullable = true)
 |-- calculated_host_listings_count: string (nullable = true)
 |-- availability_365: string (nullable = true)



In [7]:
from pyspark.sql.types import *

In [10]:
# inferSchema=True 옵션을 설정하지 않는 경우, 모든 column의 값은 str으로 로드함
# 특정 열만 선정하여 type을 지정함
# price, minimum_nights, number_of_reviews, reviews_per_month, availability_365, column은 각각 int와 float로 타입 변경
bnb = bnb.withColumn('price',bnb.price.cast(IntegerType())).withColumn('minimum_nights',bnb.minimum_nights.cast(IntegerType()))\
.withColumn('number_of_reviews',bnb.number_of_reviews.cast(IntegerType())).withColumn('reviews_per_month',bnb.reviews_per_month.cast(FloatType()))\
.withColumn('availability_365',bnb.availability_365.cast(IntegerType()))

In [12]:
bnb.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: float (nullable = true)
 |-- calculated_host_listings_count: string (nullable = true)
 |-- availability_365: integer (nullable = true)



## 1. `groupBy`

- 지정된 열의 값을 기준으로 grouping 수행
- 지정된 열의 값은 일반적으로 **categorical data**(범주형 데이터) </br>
**categorical data**: 한정적인 수의 값을 갖는 데이터 (ex> 남/여, A/B/AB/O, 10대/20대/30대/,.../90대)
- `groupBy`를 이용하여 grouping을 할 때, 반드시 해당 group의 집계된 값을 배정하기 위한 **통계/집계함수**도 함께 적용되어야 함

In [13]:
# neighbourhood_group별 숙소의 수
# brooklyn, manhattan, queens 등 각 지역에 속한 host의 수
# grouping의 기준: neighbourhood_group, 집계/통계함수: count
print(type(bnb.groupBy('neighbourhood_group')))

<class 'pyspark.sql.group.GroupedData'>


- `DataFrame.groupBy(col)`: `GroupedData` 타입을 반환
- `GroupedData.통계/집계함수()`: 각 그룹별로 통계 및 집계함수가 적용된 후, `DataFrame`으로 반환

In [25]:
neighbour_group = bnb.groupBy('neighbourhood_group').count()

In [26]:
neighbour_group.show(5)

+-------------------+-----+
|neighbourhood_group|count|
+-------------------+-----+
|         Douglaston|    1|
|             Queens| 5521|
|              Nadia|    1|
|            Midtown|    4|
|     Hell's Kitchen|    6|
+-------------------+-----+
only showing top 5 rows



In [33]:
neighbour_group.orderBy(neighbour_group['count'].desc()).show(5)
# neighbour_group.count.desc(): count를 DataFrame의 attribute로 인식하여 오류발생

+-------------------+-----+
|neighbourhood_group|count|
+-------------------+-----+
|          Manhattan|21342|
|           Brooklyn|19866|
|             Queens| 5521|
|              Bronx| 1072|
|      Staten Island|  366|
+-------------------+-----+
only showing top 5 rows



In [41]:
# neighbourhood_group 별 price 평균, 상위 5개
neighbour_avg = bnb.groupBy('neighbourhood_group').mean('price')
neighbour_avg.orderBy(neighbour_avg['avg(price)'].desc()).show(5)

+-------------------+------------------+
|neighbourhood_group|        avg(price)|
+-------------------+------------------+
|          Manhattan| 196.8079374004311|
|           Brooklyn|124.49088895600524|
|      Staten Island|115.48087431693989|
|             Queens| 99.60460061583046|
|              Bronx| 87.05970149253731|
+-------------------+------------------+
only showing top 5 rows



## 2. `agg`

- `GroupedData.agg({col: 통계함수})` 또는 `GroupedData.agg(통계함수(col))`
- 복수의 column과 통계함수를 적용할 수 있음

In [45]:
from pyspark.sql.functions import *
# column을 인자로 하는 통계함수, mean, max 등을 import

In [42]:
bnb.groupBy('neighbourhood_group').agg({'price':'mean'}).show(5)

+-------------------+------------------+
|neighbourhood_group|        avg(price)|
+-------------------+------------------+
|         Douglaston|               1.0|
|             Queens| 99.60460061583046|
|              Nadia|              NULL|
|            Midtown|               9.0|
|     Hell's Kitchen|1.3333333333333333|
+-------------------+------------------+
only showing top 5 rows



In [47]:
bnb.groupBy('neighbourhood_group').agg(mean('price')).show(5)
# 또는 bnb.groupBy('neighbourhood_group').agg(mean(bnb.price)).show(5)

+-------------------+------------------+
|neighbourhood_group|        avg(price)|
+-------------------+------------------+
|         Douglaston|               1.0|
|             Queens| 99.60460061583046|
|              Nadia|              NULL|
|            Midtown|               9.0|
|     Hell's Kitchen|1.3333333333333333|
+-------------------+------------------+
only showing top 5 rows



In [51]:
# 복수의 column에 적용
#bnb.groupBy('neighbourhood_group').agg({'price':'mean','number_of_reviews':'max'}).show(5)
bnb.groupBy('neighbourhood_group').agg(mean('price'),max('number_of_reviews')).show(5)

+-------------------+------------------+----------------------+
|neighbourhood_group|        mean_price|max(number_of_reviews)|
+-------------------+------------------+----------------------+
|         Douglaston|               1.0|                  NULL|
|             Queens| 99.60460061583046|                   629|
|              Nadia|              NULL|                    30|
|            Midtown|               9.0|                  NULL|
|     Hell's Kitchen|1.3333333333333333|                  NULL|
+-------------------+------------------+----------------------+
only showing top 5 rows



In [52]:
bnb.groupBy('neighbourhood_group').agg(mean('price').alias('mean_price'),max('number_of_reviews').alias('max_review')).show(5)

+-------------------+------------------+----------+
|neighbourhood_group|        mean_price|max_review|
+-------------------+------------------+----------+
|         Douglaston|               1.0|      NULL|
|             Queens| 99.60460061583046|       629|
|              Nadia|              NULL|        30|
|            Midtown|               9.0|      NULL|
|     Hell's Kitchen|1.3333333333333333|      NULL|
+-------------------+------------------+----------+
only showing top 5 rows



## 3. `pivot`

- **차원 전환**: 세로로 길게 늘어진 데이터를 가로로 펼쳐서, 한 눈에 비교하기 좋은 표로 변환
- **행이 될 column**: `groupBy`에 지정
- **열이 될 column**: `pivot`에 지정
- **값이 될 column**: `agg`에 지정

In [54]:
sales = spark.read.csv('C:\\datas\\3rd_week_data\\supermarket_sales.csv',inferSchema=True,header=True)
sales.printSchema()

root
 |-- Invoice ID: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer type: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Product line: string (nullable = true)
 |-- Unit price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Tax 5%: double (nullable = true)
 |-- Total: double (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Payment: string (nullable = true)
 |-- cogs: double (nullable = true)
 |-- gross margin percentage: double (nullable = true)
 |-- gross income: double (nullable = true)
 |-- Rating: double (nullable = true)



- 각 도시(`city`)에 대해서 각 `payment` 방식으로 지불한 사람의 수(`count`)를 집계
- 1. 각 도시를 기준으로 `groupBy` (**행**)
  2. `payment`를 기준으로 `pivot` (**열**)
  3. `count`로 집계 (**값**)

In [56]:
sales.groupBy('City').pivot('Payment').count().show(5)

+---------+----+-----------+-------+
|     City|Cash|Credit card|Ewallet|
+---------+----+-----------+-------+
|Naypyitaw| 124|         98|    106|
| Mandalay| 110|        109|    113|
|   Yangon| 110|        104|    126|
+---------+----+-----------+-------+



In [58]:
# pivot에 지정된 열의 경우, 해당 열의 특정 값만 지정할 수 있음
# ex> Payment의 Ewallet과 Cash만 확인하고자 함
sales.groupBy('City').pivot('Payment',['Ewallet','Cash']).count().show(5)

+---------+-------+----+
|     City|Ewallet|Cash|
+---------+-------+----+
|Naypyitaw|    106| 124|
| Mandalay|    113| 110|
|   Yangon|    126| 110|
+---------+-------+----+



- 각 `Gender`에 대해 `Ewallet`과 `Cash`로 지불한 사람들의 `Total` 값의 최대, 최소값을 보이시오
- 이 때, 최대/최소값을 포함한 열의 이름은 각각 `max_Total`, `min_Total`로 하시오

In [62]:
# Ewallet과 Cash로 지불한 남자와 여자의 최대 금액과 최소 금액
sales.groupBy('Gender').pivot('Payment',['Ewallet','Cash']).agg(min(sales.Total).alias('min_Total'),max(sales.Total).alias('max_Total')).show()

+------+-----------------+-----------------+--------------+--------------+
|Gender|Ewallet_min_Total|Ewallet_max_Total|Cash_min_Total|Cash_max_Total|
+------+-----------------+-----------------+--------------+--------------+
|Female|          26.5545|          1034.46|        13.167|       1002.12|
|  Male|           13.419|          1023.75|       10.6785|       1003.59|
+------+-----------------+-----------------+--------------+--------------+

